# About the Data

## Data Quality and Integration Issues

- Inconsistent primary keys across enrollment ranges
- Supabase RLS complications (temporarily disabled)
- Schema naming inconsistencies (for example, `School ID` vs `School_ID`)
- Case-sensitivity issues
- Fragmented time ranges
- Potential duplicate records
- Missing demographic detail
- Inconsistent weapon classification

## Imports and Supabase Connection

In [2]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv
from pathlib import Path
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf

%matplotlib inline
# Resolve the project root (one level above `/notebooks`).
project_root = Path.cwd().parent
env_path = project_root / ".env"

load_dotenv(dotenv_path=env_path, override=True)

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# Optional sanity check.
print("Loaded key starts with:", SUPABASE_KEY[:5])


Loaded key starts with: eyJhb


## Incident

In [3]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("incident")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

incident_df = pd.DataFrame(all_rows)
incident_df["Number_Victims"] = pd.to_numeric(incident_df["Number_Victims"], errors="coerce")

# Quick validation of total row count.
print (len(incident_df))
print (incident_df.head(2))


3136
     Incident_ID  Month  Day  Year                       Date  \
0  19660311NCIRC      3   11  1966  1966-03-11T00:00:00+00:00   
1  19660314TXCAW      3   14  1966  1966-03-14T00:00:00+00:00   

                             School Victims_Killed Victims_Wounded  \
0  Irwing Avenue Junior High School              0               1   
1                Carver High School              1               0   

   Number_Victims Shooter_Killed  ... Preplanned SRO_School  \
0               1              0  ...         No              
1               1              0  ...         No        Yes   

  Security_Screening     Screening_Outcome Shots_Fired School_Lockdown  \
0                                                    7                   
1       Armed Guards  Outside/Off-Property           3                   

         LAT         LNG Campus_Type Zipcode  
0  35.237069  -80.850227               28202  
1   31.57954  -97.130303               76704  

[2 rows x 50 columns]


## Shooter

In [4]:
# Read the shooter table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("shooter")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

shooter_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(shooter_df))
print (shooter_df.head(2))


3542
  Incident_ID Age Gender Race School_Affiliation Shooter_Outcome Shooter_Died  \
0                                                                               
1                                                                               

  Injury  
0         
1         


## Victim

In [5]:
# Read the victim table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("victim")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

victim_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(victim_df))
print (victim_df.head(2))


8370
     Incident_ID   Injury Gender School_Affiliation Age   Race
0  19660311NCIRC  Wounded   Male            Student  13    NaN
1  19660314TXCAW    Fatal   Male        No Relation  24  Black


## Weapon

In [6]:
# Read the weapon table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("weapon")
        .select("*")
        .order("Incident_ID")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

weapon_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(weapon_df))
print (weapon_df.head(2))


3168
     Incident_ID Weapon_Type  Weapon_Caliber Weapon_Details
0  19660311NCIRC     Handgun     .22 caliber            NaN
1  19660314TXCAW     Handgun  Service Weapon            NaN


## School Enrollment (1987-2025)

### Enrollment Anomaly Review

#### 1. Anomaly Detected: Structural Break in National Enrollment
- Longitudinal panel constructed for 1987-2025
- Aggregated national enrollment by year
- Abrupt 1.31M drop detected in 2019
- No gradual pre-trend decline
- Observation: pattern is inconsistent with historical trend behavior

#### 2. Diagnostic Investigation
Validation checks performed:
- Year-over-year change analysis
- School counts per year
- Cross-era school-universe comparison

Findings:
- School counts shift by era
- The 2016-2025 era contains fewer schools than prior periods
- Reporting universe is not consistent across datasets

#### 3. Root Cause: NCES Reporting Universe Shift
- Different download templates were used across eras
- Different school-inclusion rules were applied
- Status field availability varied (Open/Closed)
- AE or school-type inclusion may have changed

The 2019 decline reflects a structural reporting shift, not a demographic shock.

#### 4. Corrective Action
- Deleted derived tables
- Re-downloaded all eras using an identical template
- Standardized inclusion rules (for example, open schools only)
- Rebuilt the longitudinal panel with a consistent universe

Result: a comparable enrollment series across the full time span.

This demonstrates:
- Data-validation rigor
- Structural-break detection
- Source-integrity auditing
- Methodological correction

In [7]:
response = (
    supabase
    .table("national_enrollment_trend_mv")
    .select("*")
    .execute()
)

import pandas as pd

national_enrollment_trend_df = pd.DataFrame(response.data)


### Incident Rate per 100,000 Students

In [8]:
response = (
    supabase
    .table("incident_rate_per_100k")
    .select("*")
    .order("year")
    .execute()
)

import pandas as pd
df = pd.DataFrame(response.data)

print(df.head())
print(df.tail())


   year  incident_count  national_enrollment  incidents_per_100k_students
0  1987              25             37195297                     0.067213
1  1988              38             37755618                     0.100647
2  1989              20             38278250                     0.052249
3  1990              20             38858336                     0.051469
4  1991              32             39738472                     0.080526
    year  incident_count  national_enrollment  incidents_per_100k_students
34  2021             257             49057632                     0.523874
35  2022             308             49106809                     0.627204
36  2023             350             49307936                     0.709825
37  2024             337             49205891                     0.684877
38  2025             146             49033162                     0.297758


## Background Checks

In [9]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("background_check_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

background_check_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(background_check_df))
print (background_check_df.head(2))


21960
     state state_abb  fips  year year_frac        law_class law_class_subtype  \
0  Alabama        AL     1  1979         1  Castle doctrine               N/A   
1  Alabama        AL     1  1979         1   Dealer license               N/A   

  handguns_or_long_guns age_for_minimum_age_laws  law_id  
0  handgun and long gun                     #N/A  AL1004  
1               handgun                     #N/A  AL1007  


## Child Access Laws

In [10]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("child_access_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

child_access_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print (len(child_access_df))
print (child_access_df.head(2))


1710
        state state_abb  fips  year year_frac          law_class  \
0  California        CA     6  1979         0  Child access laws   
1  California        CA     6  1980         0  Child access laws   

   law_class_subtype handguns_or_long_guns age_for_minimum_age_laws  law_id  
0  Negligent storage  handgun and long gun                     #N/A  CA1011  
1  Negligent storage  handgun and long gun                     #N/A  CA1011  


## Firearm Laws

In [11]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("firearm_laws_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

firearm_laws_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(firearm_laws_df))
print(firearm_laws_df.head(2))


13095
     state state_abb  fips  year year_frac        law_class law_class_subtype  \
0  Alabama        AL     1  1979         1  Castle doctrine               N/A   
1  Alabama        AL     1  1979         1   Dealer license               N/A   

  handguns_or_long_guns age_for_minimum_age_laws  law_id  
0  handgun and long gun                     #N/A  AL1004  
1               handgun                     #N/A  AL1007  


## Firearm Sales Restrictions

In [12]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("firearm_sales_retrictions_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

firearm_sales_retrictions_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(firearm_sales_retrictions_df))
print(firearm_sales_retrictions_df.head(2))


9225
     state state_abb  fips  year year_frac                   law_class  \
0  Alabama        AL     1  1979         0  Firearm sales restrictions   
1  Alabama        AL     1  1979         0  Firearm sales restrictions   

               law_class_subtype handguns_or_long_guns  \
0  Assault weapons ban — federal               handgun   
1  Assault weapons ban — federal              long gun   

  age_for_minimum_age_laws  law_id  
0                     #N/A  AL1044  
1                     #N/A  AL1045  


## Permit To Purchase

In [13]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("permit_to_purchase_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

permit_to_purchase_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(permit_to_purchase_df))
print(permit_to_purchase_df.head(2))


3060
     state state_abb  fips  year year_frac                      law_class  \
0  Alabama        AL     1  1979         0  Local laws preempted by state   
1  Alabama        AL     1  1980         0  Local laws preempted by state   

          law_class_subtype handguns_or_long_guns age_for_minimum_age_laws  \
0  Comprehensive - punitive  handgun and long gun                     #N/A   
1  Comprehensive - punitive  handgun and long gun                     #N/A   

   law_id  
0  AL1042  
1  AL1042  


## Prohibited Possessor

In [14]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("prohibited_possessor_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

prohibited_possessor_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(prohibited_possessor_df))
print(prohibited_possessor_df.head(2))


10755
     state state_abb  fips  year year_frac             law_class  \
0  Alabama        AL     1  1979         0  Prohibited possessor   
1  Alabama        AL     1  1979         0  Prohibited possessor   

                                   law_class_subtype handguns_or_long_guns  \
0                                               Dvro  handgun and long gun   
1  Mental health : adjudicated as mentally incomp...  handgun and long gun   

  age_for_minimum_age_laws  law_id  
0                     #N/A  AL1023  
1                     #N/A  AL1024  


## Registration

In [15]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("registration_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

registration_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(registration_df))
print(registration_df.head(2))


10305
     state state_abb  fips  year year_frac        law_class law_class_subtype  \
0  Alabama        AL     1  1979         1  Castle doctrine               N/A   
1  Alabama        AL     1  1979         1   Dealer license               N/A   

  handguns_or_long_guns age_for_minimum_age_laws  law_id  
0  handgun and long gun                     #N/A  AL1004  
1               handgun                     #N/A  AL1007  


## Report Stolen Lost Firearm

In [16]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("report_stolen_lost_firearm_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

report_stolen_lost_firearm_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(report_stolen_lost_firearm_df))
print(report_stolen_lost_firearm_df.head(2))


2160
        state state_abb  fips  year year_frac  \
0  California        CA     6  1979         0   
1  California        CA     6  1979         0   

                                       law_class         law_class_subtype  \
0  Required reporting of lost or stolen firearms  Lost and stolen firearms   
1  Required reporting of lost or stolen firearms  Lost and stolen firearms   

  handguns_or_long_guns age_for_minimum_age_laws  law_id  
0               handgun                     #N/A  CA1093  
1              long gun                     #N/A  CA1094  


## Safety Training Req

In [17]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("safety_training_req_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

safety_training_req_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(safety_training_req_df))
print(safety_training_req_df.head(2))


720
        state state_abb  fips  year year_frac                 law_class  \
0  California        CA     6  1979         0  Safety training required   
1  California        CA     6  1979         0  Safety training required   

  law_class_subtype handguns_or_long_guns age_for_minimum_age_laws  law_id  
0       To purchase               handgun                     #N/A  CA1067  
1       To purchase              long gun                     #N/A  CA1068  


## Waiting Period

In [18]:
# Read the incident table in 1,000-row pages to avoid timeouts and memory spikes.
page_size = 1000
# Accumulate rows until Supabase returns an empty page.
all_rows = []
start = 0

while True:
    response = (
        supabase
        .table("waiting_period_time_series")
        .select("*")
        .order("state, year, law_id")   # Required for stable pagination.
        .range(start, start + page_size - 1)
        .execute()
    )

    data = response.data
    if not data:
        break

    all_rows.extend(data)
    start += page_size

waiting_period_df = pd.DataFrame(all_rows)

# Quick validation of total row count.
print(len(waiting_period_df))
print(waiting_period_df.head(2))


12600
     state state_abb  fips  year  year_frac        law_class  \
0  Alabama        AL     1  1979        1.0  Castle doctrine   
1  Alabama        AL     1  1979        1.0   Dealer license   

  law_class_subtype handguns_or_long_guns age_for_minimum_age_laws  law_id  
0               N/A  handgun and long gun                     #N/A  AL1004  
1               N/A               handgun                     #N/A  AL1007  
